# Parallel Computation on GPUs

In [1]:
import torch

In [2]:
import time

class Benchmark:
    def __init__(self, description="Done"):
        self.description = description
    
    def __enter__(self):
        self.start_time = time.perf_counter()
        return self
    
    def __exit__(self, *args):
        self.end_time = time.perf_counter()
        print(f'{self.description}: {self.end_time - self.start_time:.4f} sec')

In [3]:
device_count = torch.accelerator.device_count()
assert device_count > 1, "Must have multiple devices"
assert torch.cuda.is_available(), "Multiple cuda devices are required to run this tutorial"
devices = [torch.device(f"cuda:{i}") for i in range(device_count)]
print(devices)

[device(type='cuda', index=0), device(type='cuda', index=1)]


In [4]:
N = 50
M = 4000

In [5]:
def run(x):
    return [x.mm(x) for _ in range(N)]

x_gpu0 = torch.randn(size=(M, M), device=devices[0])
x_gpu1 = torch.randn(size=(M, M), device=devices[1])

In [6]:
# warm up devices
run(x_gpu0)
run(x_gpu1)
torch.cuda.synchronize(devices[0])
torch.cuda.synchronize(devices[1])

In [7]:
with Benchmark('GPU 0 time'):
    run(x_gpu0)
    torch.cuda.synchronize(devices[0])

with Benchmark('GPU 1 time'):
    run(x_gpu1)
    torch.cuda.synchronize(devices[1])

GPU 0 time: 1.5519 sec
GPU 1 time: 1.5119 sec


In [8]:
with Benchmark("GPU 0 & GPU 1"):
    run(x_gpu0)
    run(x_gpu1)
    torch.cuda.synchronize(devices[0])
    torch.cuda.synchronize(devices[1])

GPU 0 & GPU 1: 1.5678 sec


# Parallel Computation and Communication

In [9]:
def copy_to_cpu(x, non_blocking=False):
    return [y.to('cpu', non_blocking=non_blocking) for y in x]

In [10]:
with Benchmark('Run on GPU 0'):
    y = run(x_gpu0)
    torch.cuda.synchronize()

with Benchmark('Copy to CPU'):
    y_cpu = copy_to_cpu(y)
    torch.cuda.synchronize()

Run on GPU 0: 1.5745 sec
Copy to CPU: 2.8540 sec


In [11]:
with Benchmark('Run on GPU1, with blocking copy to CPU'):
    y = run(x_gpu0)
    y_cpu = copy_to_cpu(y, non_blocking=False)
    torch.cuda.synchronize()

Run on GPU1, with blocking copy to CPU: 4.8112 sec


In [12]:
with Benchmark('Run on GPU1, without blocking copy to CPU'):
    y = run(x_gpu0)
    y_cpu = copy_to_cpu(y, non_blocking=True)
    torch.cuda.synchronize()

Run on GPU1, without blocking copy to CPU: 2.4247 sec
